In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

In [3]:
x_fantoma = X_fantoma.reshape(-1)

In [4]:
def matriz_proyeccion(n, direcciones):
  res = []

  for direccion in direcciones:                                      # Por cada dirección (α, β)
    eses = []                                                        # Valores de s de esa dirección

    for i in range(n):                                               # Por cada fila
      for j in range(n):                                             # Por cada columna
        s = direccion[0] * i + direccion[1] * j                      # El s del rayo que pasa por ese píxel

        if s not in eses:                                            # Evitamos repetir rayos
          eses.append(s)

    for s in sorted(eses):                                           # Por cada rayo, de menor a mayor s
      res.append(mascara_rayo(n, direccion, s).reshape(-1))          # Agregamos su máscara vectorizada

  return np.array(res)

In [5]:
def mascara_rayo(n, direccion, s):
  res = np.zeros((n, n))                                             # Inicializamos una matriz de ceros
  for i in range(0, n):                                              # Por cada fila
    for j in range(0, n):                                            # Por cada columna
      if direccion[0] * i + direccion[1] * j == s:                   # Vemos si está en el centro
        res[i, j] = 1                                                # Modificamos

  return res

In [6]:
n=8

dir_7 = [(1,0), (0,1), (1,1), (1,-1), (2,1), (1,2), (3,1)]

A_7 = matriz_proyeccion(n, dir_7)

In [7]:
np.random.seed(42)

b_7 = A_7 @ x_fantoma

z = np.random.randn(len(b_7))

escala = np.sqrt(np.mean(b_7**2))

sigma = 0.06
eta = sigma * escala * z
ruidos = eta
b_sigma = b_7 + eta

print(f"Sigma = {sigma:.2f}")
print("Norma del ruido:", np.linalg.norm(ruidos))
print("Primeras 5 mediciones:", b_sigma[:5])

Sigma = 0.06
Norma del ruido: 0.6645884766307322
Primeras 5 mediciones: [0.03267367 1.69090503 3.3426047  2.40018432 1.98459748]


In [8]:
x_reconstruido_sigma, _, _, _ = np.linalg.lstsq(A_7, b_sigma, rcond=None)

a) Calculamos el residuo $r_\sigma = b_\sigma - A_7\hat{x}_\sigma$ y verificamos numéricamente que $||A_7^Tr_\sigma|| \approx 0$.

In [9]:
r_sigma = b_sigma - (A_7 @ x_reconstruido_sigma)

np.linalg.norm(A_7.T @ r_sigma)

np.float64(3.955109326821014e-14)

Vemos que $||A_7^Tr_\sigma|| \approx \frac{1}{3,95^{14}} \approx 0$.

b) Cuando se multiplica la matriz $A_7$ traspuesta por el vector residuo $r_\sigma$, por definición, estamos multiplicando las columnas de $A_7$ por dicho vector. Como la norma del vector resultante $A_7^Tr_\sigma$ es prácticamente cero, esto quiere decir que todas sus componentes, los productos internos columna por vector residuo, son cero. En consecuencia, $r_\sigma$ es ortogonal a cada una de las columnas de $A_7$ y, por definición de subespacio, si un vector es ortogonal a los generadores (las columnas), entonces es ortogonal a todo el espacio columna.

Podemos descomponer a cualquier $b_\sigma$ como la suma de un vector que pertenece a $Col(A_7)$ y otro que pertenece a su complemento ortogonal tal que

$$b_\sigma = p + r_\sigma$$

donde $p \in Col(A_7)$ es la proyección ortogonal de $b_\sigma$ sobre $Col(A_7)$ y $r_\sigma$ es el residuo ortogonal.

Despejando $p$ y sustituyendo $r_\sigma = b_\sigma - A_7\hat{x}_\sigma$:

$$p = b_\sigma - r_\sigma$$
$$ = b_\sigma - (b_\sigma - A_7\hat{x}_\sigma)$$

Dado que ya demostramos en el punto anterior que $r_\sigma$ es estrictamente ortogonal a $\text{Col}(A_7)$, entonces, $A_7r_\sigma$ es la proyección ortogonal de $b_\sigma$ sobre Col($A_7$) porque cumple las condiciones del Teorema de la Proyección Ortogonal:

1. $A_7r_\sigma \in Col(A_7)$ ya que es una combinación lineal de sus columnas mediante el vector de coeficientes $\hat{x}_\sigma$;
2. la diferencia entre el vector original y el residuo $r_\sigma = b_\sigma - A_7 \hat{x}_\sigma$, es ortogonal a $\text{Col}(A_7)$.

c) Las proyecciones tomadas por el escáner son las "ideales", es decir, libres de ruido. En cambio, la proyección ortogonal existe el ruido aleatorio, que por mínimos cuadrados logramos filtrar sujeto a los límites físicos del escáner, pero una parte prevalece en $\eta$.

Si sustituimos $b_\sigma = b_7 + \eta$ en la fórmula de la proyección:
$$p = \text{Proyección}(b_7 + \eta)$$

Como la proyección es un operador lineal y $b_7$ ya vive en el espacio columna de $A_7$ (su proyección sobre sí mismo es el propio $b_7$): $$p = b_7 + \text{Proyección}(\eta)$$

Por lo tanto, la diferencia matemática exacta es:
$$p - b_7 = \text{Proyección}(\eta)$$